In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Users Table
users_data = {
    'user_id': [101, 102, 103, 104, 105, 106],
    'signup_date': ['2026-01-15', '2026-01-20', '2026-02-01', '2026-02-10', '2026-03-05', '2026-03-12'],
    'region': ['North America', 'Europe', 'North America', 'Asia', 'Europe', 'North America']
}
users = pd.DataFrame(users_data)

# 2. Products Table
products_data = {
    'product_id': ['P01', 'P02', 'P03', 'P04'],
    'category': ['Electronics', 'Gaming', 'Apparel', 'Accessories'],
    'price': [120.0, 60.0, 25.0, 15.0]
}
products = pd.DataFrame(products_data)

# 3. Orders Telemetry Table (Notice some missing/unmatched IDs for testing!)
orders_data = {
    'order_id': [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'user_id': [101, 102, 101, 103, 104, 999, 102, 105],  # 999 is an orphaned user ID!
    'product_id': ['P01', 'P02', 'P03', 'P01', 'P04', 'P02', 'P99', 'P03'], # P99 is an unmapped product!
    'order_date': ['2026-01-18', '2026-01-22', '2026-02-05', '2026-02-12', '2026-02-15', '2026-03-01', '2026-03-10', '2026-03-15'],
    'quantity': [1, 2, 3, 1, 4, 1, 2, 1]
}
orders = pd.DataFrame(orders_data)

print("Users, Products, and Orders tables created successfully!")

Users, Products, and Orders tables created successfully!


In [3]:
merged_df=users.merge(orders, on='user_id',how='left').merge(products,on='product_id',how='left')

In [4]:
merged_df

,user_id,signup_date,region,order_id,product_id,order_date,quantity,category,price
0,101,2026-01-15,North America,1001.0,P01,2026-01-18,1.0,Electronics,120.0
1,101,2026-01-15,North America,1003.0,P03,2026-02-05,3.0,Apparel,25.0
2,102,2026-01-20,Europe,1002.0,P02,2026-01-22,2.0,Gaming,60.0
3,102,2026-01-20,Europe,1007.0,P99,2026-03-10,2.0,NaN,NaN
4,103,2026-02-01,North America,1004.0,P01,2026-02-12,1.0,Electronics,120.0
5,104,2026-02-10,Asia,1005.0,P04,2026-02-15,4.0,Accessories,15.0
6,105,2026-03-05,Europe,1008.0,P03,2026-03-15,1.0,Apparel,25.0
7,106,2026-03-12,North America,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
merged_df['total_spend'] = merged_df['price'] * merged_df['quantity']


In [9]:
merged_df

,user_id,signup_date,region,order_id,product_id,order_date,quantity,category,price,total_spend
0,101,2026-01-15,North America,1001.0,P01,2026-01-18,1.0,Electronics,120.0,120.0
1,101,2026-01-15,North America,1003.0,P03,2026-02-05,3.0,Apparel,25.0,75.0
2,102,2026-01-20,Europe,1002.0,P02,2026-01-22,2.0,Gaming,60.0,120.0
3,102,2026-01-20,Europe,1007.0,P99,2026-03-10,2.0,NaN,NaN,NaN
4,103,2026-02-01,North America,1004.0,P01,2026-02-12,1.0,Electronics,120.0,120.0
5,104,2026-02-10,Asia,1005.0,P04,2026-02-15,4.0,Accessories,15.0,60.0
6,105,2026-03-05,Europe,1008.0,P03,2026-03-15,1.0,Apparel,25.0,25.0
7,106,2026-03-12,North America,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
merged_df.isna().sum()

user_id        0
signup_date    0
region         0
order_id       1
product_id     1
order_date     1
quantity       1
category       2
price          2
total_spend    2
dtype: int64

In [14]:
merged_df[merged_df['price'].isna() | merged_df['region'].isna()]

,user_id,signup_date,region,order_id,product_id,order_date,quantity,category,price,total_spend
3,102,2026-01-20,Europe,1007.0,P99,2026-03-10,2.0,NaN,NaN,NaN
7,106,2026-03-12,North America,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
clean_df=merged_df.dropna(subset='total_spend')

In [16]:
clean_df

,user_id,signup_date,region,order_id,product_id,order_date,quantity,category,price,total_spend
0,101,2026-01-15,North America,1001.0,P01,2026-01-18,1.0,Electronics,120.0,120.0
1,101,2026-01-15,North America,1003.0,P03,2026-02-05,3.0,Apparel,25.0,75.0
2,102,2026-01-20,Europe,1002.0,P02,2026-01-22,2.0,Gaming,60.0,120.0
4,103,2026-02-01,North America,1004.0,P01,2026-02-12,1.0,Electronics,120.0,120.0
5,104,2026-02-10,Asia,1005.0,P04,2026-02-15,4.0,Accessories,15.0,60.0
6,105,2026-03-05,Europe,1008.0,P03,2026-03-15,1.0,Apparel,25.0,25.0


In [20]:
clean_df.groupby('user_id')['total_spend'].sum().sort_values(ascending=False)

user_id
101    195.0
102    120.0
103    120.0
104     60.0
105     25.0
Name: total_spend, dtype: float64

In [21]:
clean_df.groupby(['region','category'])['total_spend'].sum()

region         category   
Asia           Accessories     60.0
Europe         Apparel         25.0
               Gaming         120.0
North America  Apparel         75.0
               Electronics    240.0
Name: total_spend, dtype: float64